
###### 08_model_registry

###### Purpose

Register the best-performing Telco Churn classification model from MLflow into the Unity Catalog Model Registry for versioning and deployment.

###### Technologies Used

- Databricks

- Unity Catalog

- Delta Lake

- MLflow

- Scikit-learn


###### Input

- Best model metadata Delta table (BEST_MODEL_TABLE)

- Selected MLflow Run ID

- Logged MLflow model artifact (runs:/<run_id>/model)

######  Output

- Registered Unity Catalog model

- Version 1

- Registered model URI


######  Architecture

```text

BEST_MODEL_TABLE
        ↓
Retrieve best_run_id
        ↓
Construct runs:/ Model URI
        ↓
Register Model in Unity Catalog
        ↓
Create Model Version
        ↓
Ready for Serving

```

###### Section 0 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 1 : Get the best model info

In [0]:
import mlflow

best_model_df = spark.table(BEST_MODEL_TABLE)

if best_model_df.count() == 0:
    raise ValueError(
        f"No model found in {BEST_MODEL_TABLE}"
    )

best_info = best_model_df.first()



###### Section 2 : Validate run id

In [0]:
if best_info["best_run_id"] is None:
    raise ValueError(
        "best_run_id is missing."
    )

###### Section 3 : Register Model in Unity Catalog Model Registry

In [0]:
best_run_id = best_info["best_run_id"]

model_uri = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

print("=" * 60)
print("Model Registration Successful")
print("=" * 60)

print(f"Run ID          : {best_run_id}")
print(f"Model URI       : {model_uri}")
print(f"Registered Name : {REGISTERED_MODEL_NAME}")
print(f"Version         : {registered_model.version}")

###### Section 4 : Validation

In [0]:
client = mlflow.MlflowClient()

for mv in client.search_model_versions(
    f"name='{REGISTERED_MODEL_NAME}'"
):
   print("=" * 50)
   print(f"Version : {mv.version}")
   print(f"Stage   : {mv.current_stage}")
   print(f"Run ID  : {mv.run_id}")

###### Section 5 :  Review Registered Model Versions

In [0]:
#Retrieve all registered versions of the model from the Unity Catalog Model Registry
#and verify that the latest model version was successfully created.

versions = client.search_model_versions(
    f"name='{REGISTERED_MODEL_NAME}'"
)

for mv in sorted(
    versions,
    key=lambda x: int(x.version),
    reverse=True
):
    print(
        f"Version {mv.version} | "
        f"Stage={mv.current_stage} | "
        f"Run ID={mv.run_id}"
    )

##### Notebook Summary

-  Read the selected best-model metadata from Delta.

-  Retrieved the best MLflow Run ID.

-  Constructed the MLflow model URI.

-  Registered the logged model in Unity Catalog.

-  Created Version 1 of the registered model.

-  Prepared the model for serving.

#####  Key Learnings

- Difference between MLflow logged model and Unity Catalog registered model.

- Artifact_path identifies the logged model inside an MLflow run.

- Model_uri uniquely identifies the logged model to be registered.

- Model Registry creates versions automatically.

- Model registration uses run_id + artifact_path to locate the logged model.

- MLflow Logged Model
        ↓
  Model Registry
        ↓
  Serving Endpoint

- Model registration creates a versioned reference to an existing MLflow model artifact; it does not retrain the model.

###### Notebook Conclusion

- In this notebook, we built a model registration workflow that stores the best Telco Churn Classification model in Unity Catalog Model Registry with version control.

- This enables centralized model management, versioning, and controlled promotion of models for deployment.

- This will be used in the next notebook to deploy the registered model as a Databricks Model Serving Endpoint for real-time inference.

###### Next Notebook

09_model_serving

- Deploy the registered model to a Databricks Model Serving Endpoint and validate predictions using sample 